### The Game base class

In [1]:
import math
from abc import abstractmethod
from collections import namedtuple
from typing import Any


class Game:
    """A game is similar to a problem, but it has a utility for each
    state and a terminal test instead of a path cost and a goal
    test. To create a game, subclass this class and implement actions,
    result, utility, and terminal_test. You may override display and
    successors or you can inherit their default methods. You will also
    need to set the .initial attribute to the initial state; this can
    be done in the constructor."""

    initial: Any

    @abstractmethod
    def actions(self, state) -> list:
        """Return a list of the allowable moves at this point."""
        raise NotImplementedError

    @abstractmethod
    def result(self, state, move):
        """Return the state that results from making a move from a state."""
        raise NotImplementedError

    @abstractmethod
    def utility(self, state, player):
        """Return the value of this final state to player."""
        raise NotImplementedError

    def terminal_test(self, state):
        """Return True if this is a final state for the game."""
        return not self.actions(state)

    def to_move(self, state):
        """Return the player whose move it is in this state."""
        return state.to_move

    def display(self, state):
        """Print or otherwise display the state."""
        print(state)

    def __repr__(self):
        return f'<{self.__class__.__name__}>'

    def play_game(self, *players):
        """Play an n-person, move-alternating game."""
        state = self.initial
        while True:
            for player in players:
                move = player(self, state)
                state = self.result(state, move)
                if self.terminal_test(state):
                    self.display(state)
                    return self.utility(state, self.to_move(self.initial))

### Hexapawn

In [ ]:
GameState = namedtuple('GameState', 'to_move, utility, board, moves')


class Hexapawn(Game):
    """Hexapawn on an n x n board (default 3x3).

    Board is a dict {(x, y): 'W' | 'B'} with only occupied squares stored,
    1 <= x, y <= n. White's home row is y = 1 (moves toward y = n),
    Black's home row is y = n (moves toward y = 1).

    A move is a tuple ((from_x, from_y), (to_x, to_y)).

    `state.utility` is stored from WHITE's point of view: +1 = White has
    won, -1 = Black has won, 0 = game not yet decided. `Game.utility`
    flips the sign for Black.
    """

    def __init__(self, n=3):
        self.n = n
        board = {}
        for x in range(1, n + 1):
            board[(x, 1)] = 'W'
            board[(x, n)] = 'B'
        moves = self.compute_actions(board, 'W')
        self.initial = GameState(to_move='W', utility=0, board=board, moves=moves)

    # helpers
    @staticmethod
    def opponent(player):
        return 'B' if player == 'W' else 'W'

    def compute_actions(self, board, player):
        """All legal moves for `player` given `board`."""
        direction = 1 if player == 'W' else -1
        opponent = self.opponent(player)
        n = self.n
        moves = []
        for (x, y), p in board.items():
            if p != player:
                continue
            # straight advance onto an empty square
            fwd = (x, y + direction)
            if 1 <= fwd[1] <= n and fwd not in board:
                moves.append(((x, y), fwd))
            # diagonal capture
            for dx in (-1, 1):
                diag = (x + dx, y + direction)
                if 1 <= diag[0] <= n and 1 <= diag[1] <= n and board.get(diag) == opponent:
                    moves.append(((x, y), diag))
        return moves

    def compute_utility(self, board, dst, player):
        """Utility (White's perspective) after `player` just moved to `dst`."""
        n = self.n
        goal_row = n if player == 'W' else 1
        opponent = self.opponent(player)
        won = dst[1] == goal_row or not any(v == opponent for v in board.values())
        if not won:
            return 0
        return 1 if player == 'W' else -1

    def actions(self, state):
        return state.moves

    def result(self, state, move):
        if move not in state.moves:
            raise ValueError(f'illegal move {move!r} in state {state!r}')
        src, dst = move
        player = state.to_move
        opponent = self.opponent(player)

        board = state.board.copy()
        del board[src]
        board[dst] = player

        util = self.compute_utility(board, dst, player)
        moves = [] if util != 0 else self.compute_actions(board, opponent)

        # stalemate: opponent has no legal move -> mover wins
        if util == 0 and not moves:
            util = 1 if player == 'W' else -1

        return GameState(to_move=opponent, utility=util, board=board, moves=moves)

    def utility(self, state, player):
        return state.utility if player == 'W' else -state.utility

    def display(self, state):
        n = self.n
        board = state.board
        print()
        for y in range(n, 0, -1):
            print(f'{y}  ' + ' '.join(board.get((x, y), '.') for x in range(1, n + 1)))
        print('   ' + ' '.join(str(x) for x in range(1, n + 1)))
        winner = 'White' if state.utility > 0 else 'Black' if state.utility < 0 else None
        if winner:
            print(f'{winner} wins!')
        print()

### alpha-beta search

In [3]:
def alpha_beta_search(state, game):
    """Search game to determine the best move; return that move."""
    player = game.to_move(state)

    def max_value(state, alpha, beta):
        if game.terminal_test(state):
            return game.utility(state, player)
        v = -math.inf
        for a in game.actions(state):
            v = max(v, min_value(game.result(state, a), alpha, beta))
            if v >= beta:
                return v
            alpha = max(alpha, v)
        return v

    def min_value(state, alpha, beta):
        if game.terminal_test(state):
            return game.utility(state, player)
        v = math.inf
        for a in game.actions(state):
            v = min(v, max_value(game.result(state, a), alpha, beta))
            if v <= alpha:
                return v
            beta = min(beta, v)
        return v

    best_score = -math.inf
    beta = math.inf
    best_action = None
    for a in game.actions(state):
        v = min_value(game.result(state, a), best_score, beta)
        if v > best_score:
            best_score = v
            best_action = a
    return best_action


def alpha_beta_player(game, state):
    return alpha_beta_search(state, game)

### Interactive UI

In [ ]:
import ipywidgets as widgets
from IPython.display import display


PAWN_SYMBOL = {'W': '\u2659', 'B': '\u265f'}  # ♙ ♟


class InteractiveHexapawnAI:
    def __init__(self, n=3, human='W'):
        import ipywidgets as widgets  
        self.widgets = widgets        

        self.game = Hexapawn(n)
        self.n = n
        self.human = human
        self.ai = 'B' if human == 'W' else 'W'
        self.state = self.game.initial
        self.selected = None
        self.buttons = {}
        self.status_widget = None
        self.container = None
        self._build_ui()

    def _build_ui(self):
        widgets = self.widgets
        n = self.n
        grid_rows = []
        for y in range(n, 0, -1):          # top of board = row n (Black home)
            row_buttons = []
            for x in range(1, n + 1):
                btn = widgets.Button(
                    description='',
                    layout=widgets.Layout(width='70px', height='70px',
                                           border='1px solid #9ca3af'),
                    style=widgets.ButtonStyle(font_weight='bold'),
                )
                btn.square = (x, y)
                btn.on_click(self._on_click)
                self.buttons[(x, y)] = btn
                row_buttons.append(btn)
            grid_rows.append(widgets.HBox(row_buttons))

        self.status_widget = widgets.HTML()
        reset_btn = widgets.Button(description='\U0001F504 New game',
                                    button_style='success',
                                    layout=widgets.Layout(width='150px'))
        reset_btn.on_click(self.reset)

        human_name = 'White' if self.human == 'W' else 'Black'
        ai_name = 'Black' if self.ai == 'B' else 'White'
        self.container = widgets.VBox([
            widgets.HTML("<h2 style='text-align:center'>Hexapawn vs AlphaBeta AI</h2>"),
            widgets.HTML(f"<p style='text-align:center'>You: {PAWN_SYMBOL[self.human]} "
                         f"({human_name}) | AI: {PAWN_SYMBOL[self.ai]} ({ai_name})</p>"),
            self.status_widget,
            widgets.VBox(grid_rows, layout=widgets.Layout(align_items='center')),
            widgets.HTML("<br>"),
            widgets.HBox([reset_btn], layout=widgets.Layout(justify_content='center')),
        ])

        self._refresh()
        if self.state.to_move == self.ai:
            self._ai_move()

    def _on_click(self, button):
        if self.game.terminal_test(self.state) or self.state.to_move != self.human:
            return

        square = button.square

        if self.selected is None:
            if self.state.board.get(square) == self.human:
                self.selected = square
            self._refresh()
            return

        if square == self.selected:
            self.selected = None
            self._refresh()
            return

        move = (self.selected, square)
        if move in self.state.moves:
            self.state = self.game.result(self.state, move)
            self.selected = None
            self._refresh()
            if self._check_game_over():
                return
            self._ai_move()
        elif self.state.board.get(square) == self.human:
            self.selected = square
            self._refresh()
        else:
            self.selected = None
            self._refresh()

    def _ai_move(self):
        self.status_widget.value = "<h3 style='color:orange'>AI is thinking...</h3>"
        move = alpha_beta_player(self.game, self.state)
        self.state = self.game.result(self.state, move)
        self._refresh()
        self._check_game_over()

    def _check_game_over(self):
        if self.game.terminal_test(self.state):
            util = self.game.utility(self.state, self.human)
            if util == 1:
                msg = "<h3 style='color:green'>You win!</h3>"
            elif util == -1:
                msg = "<h3 style='color:red'>AI wins!</h3>"
            else:
                msg = "<h3 style='color:purple'>Draw!</h3>"
            self.status_widget.value = msg
            for btn in self.buttons.values():
                btn.disabled = True
            return True
        turn = 'Your' if self.state.to_move == self.human else "AI's"
        self.status_widget.value = f"<h3 style='color:blue'>{turn} turn</h3>"
        return False

    def _refresh(self):
        valid_targets = set()
        if self.selected is not None:
            valid_targets = {dst for src, dst in self.state.moves if src == self.selected}

        game_over = self.game.terminal_test(self.state)
        for square, btn in self.buttons.items():
            piece = self.state.board.get(square)
            btn.description = PAWN_SYMBOL.get(piece, '')
            btn.button_style = 'info' if piece == 'W' else ('danger' if piece == 'B' else '')
            btn.disabled = game_over

            if square == self.selected:
                btn.layout.border = '3px solid #2563eb'
            elif square in valid_targets:
                btn.layout.border = '3px solid #22c55e'
            else:
                btn.layout.border = '1px solid #9ca3af'

    def reset(self, button=None):
        self.state = self.game.initial
        self.selected = None
        self._refresh()
        self.status_widget.value = "<h3 style='color:blue'>Your turn</h3>"
        if self.state.to_move == self.ai:
            self._ai_move()

    def display(self):
        return self.container


game_ui = InteractiveHexapawnAI(n=3, human='W')
display(game_ui.display())